In [1]:
import os
import pandas as pd
import torch
import torchaudio
from torch.utils.data import Dataset

In [2]:
import os 
os.getcwd()

'/mnt/WorkSpace/Repos/Simulated-Vibration-Guided-Sound-Seperation/notebooks and experimentation'

In [3]:
class BirdSoundDataset(Dataset):
    def __init__(self, annotations_file, audio_dir, target_sr=16000, duration=3):
        self.annotations = pd.read_csv(annotations_file)
        self.audio_dir = audio_dir
        self.target_sr = target_sr
        self.num_samples = target_sr * duration

        self.labels = self.annotations["species"].unique()
        self.label_to_index = {label: i for i, label in enumerate(self.labels)}

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        path = self._get_audio_sample_path(index)
        label = self._get_audio_sample_label(index)

        signal, sr = torchaudio.load(path)

        # Convert to mono
        signal = torch.mean(signal, dim=0, keepdim=True)

        # Resample
        if sr != self.target_sr:
            resampler = torchaudio.transforms.Resample(sr, self.target_sr)
            signal = resampler(signal)

        # Fix length (random crop or pad)
        signal = self._fix_length(signal)

        return signal, label

    def _fix_length(self, signal):
        if signal.shape[1] > self.num_samples:
            start = torch.randint(0, signal.shape[1] - self.num_samples, (1,))
            signal = signal[:, start:start + self.num_samples]
        else:
            pad = self.num_samples - signal.shape[1]
            signal = F.pad(signal, (0, pad))
        return signal

    def _get_audio_sample_path(self, index):
        filename = self.annotations.iloc[index]["filename"]
        return os.path.join(self.audio_dir, filename)

    def _get_audio_sample_label(self, index):
        species = self.annotations.iloc[index]["species"]
        return self.label_to_index[species]

In [4]:
import torch.nn as nn

class AudioClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=16000,
            n_mels=64
        )

        self.db = torchaudio.transforms.AmplitudeToDB()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        # x: (B, 1, T)

        x = self.mel(x)       # (B, 1, n_mels, time)
        x = self.db(x)

        x = self.cnn(x)       # (B, 64, 1, 1)
        x = x.view(x.size(0), -1)

        x = self.fc(x)
        return x

In [18]:
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

dataset = BirdSoundDataset(
    "/mnt/WorkSpace/Repos/Simulated-Vibration-Guided-Sound-Seperation/notebooks and experimentation/data/dataset_1/bird_songs_metadata.csv",
    "/mnt/WorkSpace/Repos/Simulated-Vibration-Guided-Sound-Seperation/notebooks and experimentation/data/dataset_1/wavfiles/"
)

dataloader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=0)

model = AudioClassifier(num_classes=len(dataset.labels)).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    total_loss = 0
    model.train()

    for signals, labels in dataloader:
        signals = signals.to(device)
        labels = labels.to(device)

        outputs = model(signals)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 68.7353
Epoch 2, Loss: 64.6735
Epoch 3, Loss: 60.5135
Epoch 4, Loss: 55.7290
Epoch 5, Loss: 49.7452
